In [4]:
import os
import argparse
import yaml
import sys
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd

from models.sit_recon_encoder import SiT
from utils.utils import save_gifti, get_dataloaders

In [20]:
save_folder = "/ceph/chpc/shared/janine_bijsterbosch_group/lucas_scratch/SiT_structural_encoder/demeaning_track"

if not os.path.exists(save_folder):
    os.mkdir(save_folder)
    print('Creating folder: {}'.format(save_folder))


In [5]:
data_path = "/ceph/chpc/shared/janine_bijsterbosch_group/lucas_scratch/HCP_YA"
patch_extraction_path = "/ceph/chpc/shared/janine_bijsterbosch_group/lucas_scratch/SiT_structural_encoder/patch_extraction"
indices_mesh_triangles=pd.read_csv(f'{patch_extraction_path}/triangle_indices_ico_6_sub_ico_2.csv')


train_data = np.load(os.path.join(data_path,'train_L_data.npy'))
val_data = np.load(os.path.join(data_path,'validation_L_data.npy'))
test_data = np.load(os.path.join(data_path,'test_L_data.npy'))

train_data[0][0:10]

array([[[2.39331102, 2.76025987, 3.1025095 , ..., 2.01600504,
         2.44600606, 2.93371177],
        [2.67760611, 2.75735283, 3.48165965, ..., 3.52190375,
         3.19965482, 3.557863  ],
        [3.68187165, 3.39522505, 3.74645138, ..., 3.79569554,
         3.63911915, 3.1552422 ],
        ...,
        [2.05580807, 2.84470749, 2.9256444 , ..., 2.48833942,
         2.59975028, 2.09663677],
        [2.51729345, 2.29431343, 2.22710323, ..., 2.13507891,
         2.09677315, 1.85305929],
        [2.25141835, 2.39028645, 2.26396537, ..., 3.3115623 ,
         2.7884779 , 2.67634726]]], shape=(1, 320, 153))

In [ ]:
tpi = get_dataloaders(data_path, )

In [22]:
idx = 5

In [24]:
def save_that_subject(data, idx, name, folder=save_folder):
    input = data[idx]
    input = input[np.newaxis, :, :, :]
    
    np.save(f"{save_folder}/{name}", input)

    original_sphere = np.zeros((40962,1),dtype=np.float32)
    new_inputs = np.transpose(input,(0,2,1,3))

    for i in range(320):
        indices_to_extract = indices_mesh_triangles[str(i)].values
        original_sphere[indices_to_extract,:] = new_inputs[0,i,:,:].transpose()

    save_gifti(original_sphere, os.path.join(folder,f"{name}.L.shape.gii"))

In [25]:
save_that_subject(test_data, idx, f"pre_demeaning_idx{idx}")

def subject_normalize(x, eps=1e-8):
    ## take the average value, std per subject, and normalize (work with intra-subject vertices)
    mean = x.mean(axis=(1,2,3), keepdims=True)
    std = x.std(axis=(1,2,3), keepdims=True)
    return (x - mean) / (std + eps)

# per-subject normalization
train_data = subject_normalize(train_data)
val_data = subject_normalize(val_data)
test_data = subject_normalize(test_data)

save_that_subject(test_data, idx, f"per_subject_normalized_idx{idx}")

## take the average of each vertex across subjects (work with inter-subject vertices)
# [B, C, N, V] --> [1, C, N, V]
vertex_wise_map = train_data.mean(axis = 0, keepdims=True)
print(f"vertex mean map dims: {vertex_wise_map.shape}")

# demeaning here
train_data = (train_data - vertex_wise_map)
val_data = (val_data - vertex_wise_map)
test_data = (test_data - vertex_wise_map)

save_that_subject(test_data, idx, f"full_demeaned_idx{idx}")

vertex mean map dims: (1, 1, 320, 153)
